In [0]:
MY_ID = "Musarraf_7045"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.capstone_Musarraf_7045;
CREATE VOLUME IF NOT EXISTS workspace.capstone_Musarraf_7045.raw;

In [0]:
import pyspark.sql.functions as F; SEED=10
PM="array('UPI','CARD','NETBANKING','WALLET')"
b=spark.range(41000).selectExpr('id',"format_string('TXN%07d',id) txn_id",
  "format_string('M%03d',id%200) merchant_id","'INR' currency",
  f"element_at({PM},cast(id%4+1 as int)) payment_method",
  "if(id between 3000 and 3209,round(1500+rand(11)*8000,2),"
  "round(exp(6.8+randn(10)*0.7),2)) amount",
  "case when id between 1000 and 1599 then cast(rand(12)*30 as int)*1440+360"
  "+cast(rand(13)*360 as int) when id between 1600 and 1899 then"
  " cast(rand(14)*30 as int)*1440+1260+cast(rand(15)*179 as int)"
  " else cast(rand(16)*43200 as int) end mins")
b=b.selectExpr('id','txn_id','merchant_id','currency','payment_method','amount',
  "timestampadd(minute,mins,timestamp'2025-06-01 00:00:00') txn_ts")
g=b.selectExpr('id','txn_id','txn_ts','merchant_id',
  "if(id between 2000 and 2349,'NA',cast(amount as string)) amount",
  'currency','payment_method',"'SUCCESS' status")
l=b.selectExpr('id',"if(id between 5000 and 5074,concat(txn_id,'  '),txn_id) txn_id",
  "if(id between 1000 and 1899,timestampadd(hour,6,txn_ts),txn_ts) txn_ts",'merchant_id',
  "case when id<400 then cast(round(amount*(1-0.0177),2) as string)"
  " when id<590 then cast(round(amount+0.01,2) as string)"
  " when id<640 then cast(round(amount*10,2) as string)"
  " when id between 3000 and 3209 then format_number(amount,2)"
  " else cast(amount as string) end amount",'currency','payment_method',"'POSTED' status")
gx=g.filter('id between 8000 and 8207').withColumn('txn_id',
  F.format_string('TXNG%06d',F.col('id')-8000))
lx=l.filter('id between 9000 and 9192').withColumn('txn_id',
  F.format_string('TXNL%06d',F.col('id')-9000))
go=g.unionByName(g.filter('id between 6000 and 6259')).unionByName(gx).drop('id')
lo=l.unionByName(l.filter('id between 7000 and 7129')).unionByName(lx).drop('id')
for d,n in [(go,'gateway'),(lo,'ledger')]:
  d.write.mode('overwrite').option('header',True) \
    .option('ignoreTrailingWhiteSpace',False).csv(f'{VOL}/raw/{n}')

In [0]:
for name in ["gateway", "ledger"]:
    raw = spark.read.option("header", True).csv(f"{VOL}/raw/{name}")
    bronze = (raw
        .select([F.col(c).cast("string") for c in raw.columns])
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_row_hash", F.sha2(F.concat_ws("||", *raw.columns), 256))
    )
    bronze.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.bronze_{name}")
    print(name, bronze.count())

gateway 41468
ledger 41323
